In [13]:
import numpy as np
import re
from fractions import Fraction

def print_matrix(mat):
    """Prints the matrix in a clean, readable format without verbose Fraction wrappers."""
    cleaned_rows = []
    for row in mat:
        cleaned_row = []
        for val in row:
            f = Fraction(val).limit_denominator()
            if f.denominator == 1:
                cleaned_row.append(f.numerator)
            else:
                cleaned_row.append(f"{f.numerator}/{f.denominator}")
        cleaned_rows.append(cleaned_row)

    for r in cleaned_rows:
        print(r)
    print()

def perform_row_operation(A, op_str):
    op_str = op_str.replace(" ", "")

    # 1. Check for Row Swap: e.g., R1<->R2
    swap_match = re.match(r"R(\d+)<->R(\d+)", op_str)
    if swap_match:
        r1 = int(swap_match.group(1)) - 1
        r2 = int(swap_match.group(2)) - 1

        # Validate row indices are within bounds
        if not (0 <= r1 < len(A) and 0 <= r2 < len(A)):
            raise ValueError(f"Row index out of range. Matrix has {len(A)} rows.")

        new_A = A.copy()
        new_A[[r1, r2]] = new_A[[r2, r1]]
        return new_A

    # 2. Check for Row Replacement/Scaling: e.g., R2->R2-3*R1, R1->3*R1
    match = re.match(r"R(\d+)->(.*)", op_str)
    if not match:
        raise ValueError(
            "Invalid operation format! "
            "Use '<->' for swaps (e.g., R1 <-> R2) "
            "or '->' for replacement/scaling (e.g., R2 -> R2 - 3*R1)."
        )

    target_idx = int(match.group(1)) - 1
    expr = match.group(2)

    # Validate target row index is within bounds
    if not (0 <= target_idx < len(A)):
        raise ValueError(f"Target row index out of range. Matrix has {len(A)} rows.")

    # Check if user accidentally wrote something invalid like just another row name without an operator
    if re.fullmatch(r"R\d+", expr):
        raise ValueError(
            f"Invalid replacement: '{op_str}'. "
            "Did you mean to use a swap '<->' instead of '->'?"
        )

    new_A = A.copy()

    def replace_row(m):
        r_num = int(m.group(1)) - 1
        if not (0 <= r_num < len(A)):
            raise ValueError(f"Referenced row R{r_num + 1} is out of range. Matrix has {len(A)} rows.")
        return f"A[{r_num}]"

    python_expr = re.sub(r"R(\d+)", replace_row, expr)

    try:
        new_A[target_idx] = eval(python_expr, {"A": A, "np": np, "Fraction": Fraction})
    except Exception as e:
        raise ValueError(f"Error evaluating expression: {e}. Check your math syntax.")

    return new_A

# --- MAIN EXECUTION ---
print("--- Interactive Elementary Row Operation Tool ---")
rows = int(input("Enter the number of rows for Matrix A: "))
cols = int(input("Enter the number of columns for Matrix A: "))

print(f"\nEnter the entries row by row (separate elements using spaces, e.g., 3 3 2):")
A_list = []
for i in range(rows):
    row_vals = [Fraction(x) for x in input(f"Row {i+1} (space-separated): ").strip().split()]
    A_list.append(row_vals)

original_matrix = np.array(A_list, dtype=object)
current_matrix = original_matrix.copy()

# History tracking logs
history = []

print("\nOriginal Matrix A:")
print_matrix(current_matrix)

# --- CONTINUOUS INTERACTIVE LOOP ---
step_count = 1
while True:
    print(f"Examples: R1 <-> R2 (Swap) | R1 -> 3*R1 (Scaling) | R2 -> R2 - 3*R1 (Replacement)")
    op_input = input(f"Enter row operation (space-separated or type 'Stop' to finish): ").strip()

    if op_input.lower() == 'stop':
        break

    try:
        updated_matrix = perform_row_operation(current_matrix, op_input)

        # Log the step
        history.append({
            "step": step_count,
            "operation": op_input,
            "matrix": updated_matrix.copy()
        })

        print(f"\n--- Output after Step {step_count}: {op_input} ---")
        print_matrix(updated_matrix)

        # Update current matrix for the next iteration
        current_matrix = updated_matrix
        step_count += 1

    except ValueError as ve:
        print(f"\n⚠️ WARNING: {ve}\nPlease check your input and try again.\n")
    except Exception as e:
        print(f"\n⚠️ WARNING: An unexpected error occurred: {e}\nPlease try again.\n")

# --- FINAL SUMMARY REPORT ---
print("\n" + "="*40)
print("       FINAL EXECUTION SUMMARY       ")
print("="*40)

print("1. Original Matrix:")
print_matrix(original_matrix)

if history:
    print("2. Intermediate Reduced Matrices:")
    for item in history:
        print(f"--- Step {item['step']} (Operation: {item['operation']}) ---")
        print_matrix(item['matrix'])

    print("3. Final Reduced Matrix:")
    print_matrix(current_matrix)
else:
    print("No row operations were performed.")

--- Interactive Elementary Row Operation Tool ---
Enter the number of rows for Matrix A: 4
Enter the number of columns for Matrix A: 3

Enter the entries row by row (separate elements using spaces, e.g., 3 3 2):
Row 1 (space-separated): 3 3 2
Row 2 (space-separated): 1 2 0
Row 3 (space-separated): 0 10 3
Row 4 (space-separated): 2 -3 -1

Original Matrix A:
[3, 3, 2]
[1, 2, 0]
[0, 10, 3]
[2, -3, -1]

Examples: R1 <-> R2 (Swap) | R1 -> 3*R1 (Scaling) | R2 -> R2 - 3*R1 (Replacement)
Enter row operation (space-separated or type 'Stop' to finish): R1 -> R2

⚠️ WARNING: Invalid replacement: 'R1->R2'. Did you mean to use a swap '<->' instead of '->'?
Please check your input and try again.

Examples: R1 <-> R2 (Swap) | R1 -> 3*R1 (Scaling) | R2 -> R2 - 3*R1 (Replacement)
Enter row operation (space-separated or type 'Stop' to finish): R1 <-> R2

--- Output after Step 1: R1 <-> R2 ---
[1, 2, 0]
[3, 3, 2]
[0, 10, 3]
[2, -3, -1]

Examples: R1 <-> R2 (Swap) | R1 -> 3*R1 (Scaling) | R2 -> R2 - 3*R1 